In [1]:
import os
import sys
import json
import logging
import sqlite3
import asyncio
from pathlib import Path
from typing import Optional, Literal

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException
from sqlglot import parse, exp

from mcp.server.fastmcp import FastMCP
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
SERVER_DIR = PROJECT_ROOT / "mcp_servers" / "writer"
AGENT_DIR = PROJECT_ROOT / "agents"

DATABASE_PATH = DATA_DIR / "verisql_demo.db"
SCHEMA_SERVER_PATH = PROJECT_ROOT / "mcp_servers" / "schema" / "server.py"
WRITER_SERVER_PATH = SERVER_DIR / "server.py"
WRITER_AGENT_PATH = AGENT_DIR / "writer.py"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SERVER_DIR.mkdir(parents=True, exist_ok=True)
AGENT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Database exists:", DATABASE_PATH.exists())
print("Schema MCP exists:", SCHEMA_SERVER_PATH.exists())
print("Writer MCP directory:", SERVER_DIR.resolve())
print("Writer Agent directory:", AGENT_DIR.resolve())

C:\Users\monis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL
Database exists: True
Schema MCP exists: True
Writer MCP directory: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\mcp_servers\writer
Writer Agent directory: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\agents


In [2]:
assert DATABASE_PATH.exists(), "Demo database not found."
assert SCHEMA_SERVER_PATH.exists(), "Schema MCP server not found."

connection = sqlite3.connect(DATABASE_PATH)
tables = connection.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name").fetchall()
connection.close()

print("Database: VERIFIED")
print("Schema MCP server: VERIFIED")
print("Tables:", [table[0] for table in tables])
print("Total tables:", len(tables))

Database: VERIFIED
Schema MCP server: VERIFIED
Tables: ['departments', 'employee_projects', 'employees', 'projects']
Total tables: 4


In [3]:
from dotenv import load_dotenv

ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY missing. Check your .env file.")

print("Groq API key loaded successfully.")

Groq API key loaded successfully.


In [5]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0, api_key=GROQ_API_KEY)

response = llm.invoke("Reply with exactly: Groq connection successful.")

print(response.content)

Groq connection successful


In [6]:
class SQLGeneration(BaseModel):
    sql_query: str = Field(description="Generated SQLite SELECT query")
    explanation: str = Field(description="Brief explanation of the SQL query")
    tables_used: list[str] = Field(description="Tables referenced in the query")
    assumptions: list[str] = Field(description="Assumptions made while generating SQL")

print("SQLGeneration schema created successfully.")

SQLGeneration schema created successfully.


In [7]:
sql_parser = PydanticOutputParser(pydantic_object=SQLGeneration)

format_instructions = sql_parser.get_format_instructions()

print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"sql_query": {"description": "Generated SQLite SELECT query", "title": "Sql Query", "type": "string"}, "explanation": {"description": "Brief explanation of the SQL query", "title": "Explanation", "type": "string"}, "tables_used": {"description": "Tables referenced in the query", "items": {"type": "string"}, "title": "Tables Used", "type": "array"}, "assumptions": {"description": "Assumptions made while generating SQL", "items": {"type": "string"}, "title": "Assumptions", "type": "array"}}, "required": ["sql_query", "explanation

In [8]:
async def get_database_context(question: str, db_id: str = "verisql_demo"):
    server_params = StdioServerParameters(command=sys.executable, args=[str(SCHEMA_SERVER_PATH)])

    with open(PROJECT_ROOT / "schema_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                relevant = await session.call_tool("find_relevant_columns", {"db_id": db_id, "question": question})
                schema = await session.call_tool("get_schema", {"db_id": db_id})
                return {"relevant_columns": [item.text for item in relevant.content], "schema": schema.content[0].text}

print("Schema MCP connection function created.")

Schema MCP connection function created.


In [9]:
test_question = "Find the names of employees working in the Engineering department."

schema_context = await get_database_context(test_question)

print("RELEVANT COLUMNS:")
for item in schema_context["relevant_columns"]:
    print(item)

print("\nDATABASE SCHEMA:")
print(schema_context["schema"])

RELEVANT COLUMNS:
{
  "table": "departments",
  "column": "department_name",
  "score": 13
}
{
  "table": "employees",
  "column": "employee_name",
  "score": 8
}
{
  "table": "departments",
  "column": "department_id",
  "score": 5
}
{
  "table": "employee_projects",
  "column": "employee_id",
  "score": 5
}
{
  "table": "employees",
  "column": "department_id",
  "score": 5
}
{
  "table": "employees",
  "column": "employee_id",
  "score": 5
}
{
  "table": "projects",
  "column": "project_name",
  "score": 3
}

DATABASE SCHEMA:
{
  "departments": {
    "columns": [
      {
        "name": "department_id",
        "type": "INTEGER",
        "primary_key": true,
        "nullable": true,
        "sensitive": false
      },
      {
        "name": "department_name",
        "type": "TEXT",
        "primary_key": false,
        "nullable": false,
        "sensitive": false
      },
      {
        "name": "location",
        "type": "TEXT",
        "primary_key": false,
        "nullable"

In [10]:
writer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are the Writer Agent in VeriSQL, a Text-to-SQL system.

Generate accurate SQLite SELECT queries using only the provided database schema.

Rules:
1. Use only existing tables and columns from the schema.
2. Follow foreign-key relationships when creating JOINs.
3. Never invent table names, column names, or relationships.
4. Do not access sensitive columns.
5. Generate only read-only SELECT queries.
6. Use relevant columns as guidance, but consult the complete schema.
7. If the question is ambiguous, state your assumptions.
8. Return output strictly in the requested JSON format.

{format_instructions}"""),
    ("human", """User question:
{question}

Relevant columns:
{relevant_columns}

Complete database schema:
{schema}

Generate the SQL query.""")
])

print("Writer prompt created successfully.")

Writer prompt created successfully.


In [11]:
writer_chain = writer_prompt | llm | sql_parser

async def generate_sql(question: str, db_id: str = "verisql_demo") -> SQLGeneration:
    context = await get_database_context(question, db_id)
    return writer_chain.invoke({
        "question": question,
        "relevant_columns": "\n".join(context["relevant_columns"]),
        "schema": context["schema"],
        "format_instructions": format_instructions
    })

print("Writer Agent chain created successfully.")

Writer Agent chain created successfully.


In [12]:
test_question = "Find the names of employees working in the Engineering department."

generated_result = await generate_sql(test_question)

print("GENERATED SQL:")
print(generated_result.sql_query)

print("\nEXPLANATION:")
print(generated_result.explanation)

print("\nTABLES USED:")
print(generated_result.tables_used)

print("\nASSUMPTIONS:")
print(generated_result.assumptions)

GENERATED SQL:
SELECT e.employee_name FROM employees e JOIN departments d ON e.department_id = d.department_id WHERE d.department_name = 'Engineering';

EXPLANATION:
Joins the employees table with departments on department_id and filters for the Engineering department, returning the employee names.

TABLES USED:
['employees', 'departments']

ASSUMPTIONS:
[]


In [13]:
def validate_generated_sql(sql_query: str) -> dict:
    try:
        statements = parse(sql_query, read="sqlite")
        if len(statements) != 1 or statements[0] is None:
            return {"valid": False, "reason": "Only one SQL statement is allowed."}

        statement = statements[0]
        if not isinstance(statement, (exp.Select, exp.Union, exp.Intersect, exp.Except)):
            if not isinstance(statement, exp.Query):
                return {"valid": False, "reason": "Only read-only SELECT queries are allowed."}

        if any(isinstance(node, (exp.Insert, exp.Update, exp.Delete, exp.Create, exp.Drop, exp.Alter, exp.Command)) for node in statement.walk()):
            return {"valid": False, "reason": "Write or schema-modification operations are not allowed."}

        return {"valid": True, "reason": "SQL passed structural validation."}

    except Exception as error:
        return {"valid": False, "reason": str(error)}

validation_result = validate_generated_sql(generated_result.sql_query)
print(validation_result)

{'valid': True, 'reason': 'SQL passed structural validation.'}


In [14]:
test_queries = {
    "valid_select": "SELECT employee_name FROM employees",
    "valid_join": "SELECT e.employee_name FROM employees e JOIN departments d ON e.department_id = d.department_id",
    "insert_attack": "INSERT INTO employees (employee_name) VALUES ('Test')",
    "delete_attack": "DELETE FROM employees",
    "multiple_statements": "SELECT * FROM employees; DROP TABLE employees"
}

for test_name, query in test_queries.items():
    result = validate_generated_sql(query)
    print(f"{test_name}: {result}")

valid_select: {'valid': True, 'reason': 'SQL passed structural validation.'}
valid_join: {'valid': True, 'reason': 'SQL passed structural validation.'}
insert_attack: {'valid': False, 'reason': 'Only read-only SELECT queries are allowed.'}
delete_attack: {'valid': False, 'reason': 'Only read-only SELECT queries are allowed.'}
multiple_statements: {'valid': False, 'reason': 'Only one SQL statement is allowed.'}


In [15]:
def execute_generated_sql(sql_query: str):
    validation = validate_generated_sql(sql_query)
    if not validation["valid"]:
        raise ValueError(validation["reason"])

    db_uri = f"file:{DATABASE_PATH.resolve().as_posix()}?mode=ro"
    with sqlite3.connect(db_uri, uri=True) as connection:
        connection.execute("PRAGMA query_only = ON")
        cursor = connection.execute(sql_query)
        columns = [item[0] for item in cursor.description]
        rows = cursor.fetchmany(20)

    return {"columns": columns, "rows": rows}

execution_result = execute_generated_sql(generated_result.sql_query)

print("COLUMNS:", execution_result["columns"])
print("RESULTS:")
for row in execution_result["rows"]:
    print(row)

COLUMNS: ['employee_name']
RESULTS:
('Arun',)
('Rahul',)
('Arun',)
('Priya',)


In [16]:
with sqlite3.connect(f"file:{DATABASE_PATH.resolve().as_posix()}?mode=ro", uri=True) as connection:
    print("DEPARTMENTS:")
    for row in connection.execute("SELECT * FROM departments"):
        print(row)

    print("\nEMPLOYEES:")
    for row in connection.execute("SELECT employee_id, employee_name, department_id FROM employees"):
        print(row)

DEPARTMENTS:
(1, 'Engineering', 'Bangalore')
(2, 'Analytics', 'Hyderabad')
(3, 'Human Resources', 'Delhi')

EMPLOYEES:
(1, 'Arun', 1)
(2, 'Priya', 2)
(3, 'Rahul', 1)
(4, 'Sneha', 3)
(5, 'Kiran', 2)
(101, 'Arun', 1)
(102, 'Priya', 1)
(103, 'Rahul', 2)
(104, 'Sneha', 2)
(105, 'Kiran', 3)


In [17]:
test_question_2 = "Find the average salary of employees in each department."

generated_result_2 = await generate_sql(test_question_2)

print("GENERATED SQL:")
print(generated_result_2.sql_query)

print("\nEXPLANATION:")
print(generated_result_2.explanation)

print("\nTABLES USED:")
print(generated_result_2.tables_used)

print("\nVALIDATION:")
print(validate_generated_sql(generated_result_2.sql_query))

GENERATED SQL:
SELECT d.department_name, AVG(e.salary) AS avg_salary FROM employees e JOIN departments d ON e.department_id = d.department_id GROUP BY d.department_id, d.department_name;

EXPLANATION:
Calculates the average salary for each department by joining employees with departments and grouping by department.

TABLES USED:
['employees', 'departments']

VALIDATION:
{'valid': True, 'reason': 'SQL passed structural validation.'}


In [18]:
execution_result_2 = execute_generated_sql(generated_result_2.sql_query)

print("COLUMNS:", execution_result_2["columns"])
print("RESULTS:")
for row in execution_result_2["rows"]:
    print(row)

COLUMNS: ['department_name', 'avg_salary']
RESULTS:
('Engineering', 80000.0)
('Analytics', 75750.0)
('Human Resources', 75000.0)


In [19]:
async def repair_sql(question: str, previous_sql: str, evaluator_feedback: str, db_id: str = "verisql_demo") -> SQLGeneration:
    context = await get_database_context(question, db_id)
    repair_input = {
        "question": question,
        "relevant_columns": "\n".join(context["relevant_columns"]),
        "schema": context["schema"],
        "format_instructions": format_instructions
    }
    repair_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are the VeriSQL Writer Agent. Repair the previous SQL using the evaluator's feedback.
Use only the supplied SQLite schema. Generate one read-only SELECT query.
Do not repeat the identified mistake. Return the required JSON format.
{format_instructions}"""),
        ("human", """Original question: {question}
Previous SQL: {previous_sql}
Evaluator feedback: {evaluator_feedback}
Relevant columns: {relevant_columns}
Database schema: {schema}
Generate corrected SQL.""")
    ])
    repair_chain = repair_prompt | llm | sql_parser
    return repair_chain.invoke({**repair_input, "previous_sql": previous_sql, "evaluator_feedback": evaluator_feedback})

print("Writer Agent repair function created successfully.")

Writer Agent repair function created successfully.


In [20]:
incorrect_sql = "SELECT e.name FROM employees e WHERE e.department_id = 1"

evaluator_feedback = """
Validation failed: Unknown column 'name' in employees table.
The correct employee name column is 'employee_name'.
Replace the invalid column and preserve the original question's intent.
"""

repaired_result = await repair_sql(
    question="Find the names of employees working in the Engineering department.",
    previous_sql=incorrect_sql,
    evaluator_feedback=evaluator_feedback
)

print("PREVIOUS SQL:")
print(incorrect_sql)

print("\nREPAIRED SQL:")
print(repaired_result.sql_query)

print("\nEXPLANATION:")
print(repaired_result.explanation)

PREVIOUS SQL:
SELECT e.name FROM employees e WHERE e.department_id = 1

REPAIRED SQL:
SELECT e.employee_name
FROM employees e
JOIN departments d ON e.department_id = d.department_id
WHERE d.department_name = 'Engineering';

EXPLANATION:
The query selects employee names from the employees table, joining with departments to filter for those in the Engineering department. The column name was corrected to employee_name as per the schema.


In [21]:
repair_validation = validate_generated_sql(repaired_result.sql_query)

print("VALIDATION:", repair_validation)

if repair_validation["valid"]:
    repair_execution = execute_generated_sql(repaired_result.sql_query)
    print("\nREPAIRED QUERY RESULTS:")
    for row in repair_execution["rows"]:
        print(row)

VALIDATION: {'valid': True, 'reason': 'SQL passed structural validation.'}

REPAIRED QUERY RESULTS:
('Arun',)
('Rahul',)
('Arun',)
('Priya',)


In [22]:
REPAIR_LOG_PATH = DATA_DIR / "writer_repair_logs.jsonl"

print("Repair log path:", REPAIR_LOG_PATH.resolve())
print("Existing log:", REPAIR_LOG_PATH.exists())

Repair log path: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\data\writer_repair_logs.jsonl
Existing log: False


In [23]:
def log_repair(question: str, initial_sql: str, evaluator_feedback: str, repaired_sql: str, initial_validation: dict, final_validation: dict):
    log_entry = {
        "question": question,
        "initial_sql": initial_sql,
        "evaluator_feedback": evaluator_feedback,
        "repaired_sql": repaired_sql,
        "initial_validation": initial_validation,
        "final_validation": final_validation,
        "repair_success": final_validation["valid"]
    }

    with open(REPAIR_LOG_PATH, "a", encoding="utf-8") as file:
        file.write(json.dumps(log_entry) + "\n")

    print("Repair attempt logged successfully.")

print("Repair logging function created.")

Repair logging function created.


In [24]:
initial_validation = validate_generated_sql(incorrect_sql)

log_repair(
    question="Find the names of employees working in the Engineering department.",
    initial_sql=incorrect_sql,
    evaluator_feedback=evaluator_feedback,
    repaired_sql=repaired_result.sql_query,
    initial_validation=initial_validation,
    final_validation=repair_validation
)

print("Total logged attempts:", sum(1 for _ in open(REPAIR_LOG_PATH, encoding="utf-8")))

Repair attempt logged successfully.
Total logged attempts: 1


In [25]:
with open(REPAIR_LOG_PATH, "r", encoding="utf-8") as file:
    repair_logs = [json.loads(line) for line in file if line.strip()]

print(json.dumps(repair_logs[-1], indent=4))

{
    "question": "Find the names of employees working in the Engineering department.",
    "initial_sql": "SELECT e.name FROM employees e WHERE e.department_id = 1",
    "evaluator_feedback": "\nValidation failed: Unknown column 'name' in employees table.\nThe correct employee name column is 'employee_name'.\nReplace the invalid column and preserve the original question's intent.\n",
    "repaired_sql": "SELECT e.employee_name\nFROM employees e\nJOIN departments d ON e.department_id = d.department_id\nWHERE d.department_name = 'Engineering';",
    "initial_validation": {
        "valid": true,
        "reason": "SQL passed structural validation."
    },
    "final_validation": {
        "valid": true,
        "reason": "SQL passed structural validation."
    },
    "repair_success": true
}


In [26]:
total_attempts = len(repair_logs)
successful_repairs = sum(log["repair_success"] for log in repair_logs)
repair_success_rate = (successful_repairs / total_attempts) * 100 if total_attempts else 0

print("Total repair attempts:", total_attempts)
print("Successful repairs:", successful_repairs)
print(f"Repair success rate: {repair_success_rate:.2f}%")

Total repair attempts: 1
Successful repairs: 1
Repair success rate: 100.00%


In [27]:
writer_code = r'''
import os
import json
import sqlite3
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from sqlglot import parse, exp
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"
SCHEMA_SERVER_PATH = PROJECT_ROOT / "mcp_servers" / "schema" / "server.py"
REPAIR_LOG_PATH = DATA_DIR / "writer_repair_logs.jsonl"

load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY missing from .env")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0, api_key=GROQ_API_KEY)

class SQLGeneration(BaseModel):
    sql_query: str = Field(description="Generated SQLite SELECT query")
    explanation: str = Field(description="Brief explanation of the SQL query")
    tables_used: list[str] = Field(description="Tables referenced in the query")
    assumptions: list[str] = Field(description="Assumptions made while generating SQL")

sql_parser = PydanticOutputParser(pydantic_object=SQLGeneration)
format_instructions = sql_parser.get_format_instructions()

async def get_database_context(question: str, db_id: str = "verisql_demo"):
    server_params = StdioServerParameters(command=os.sys.executable, args=[str(SCHEMA_SERVER_PATH)])
    with open(PROJECT_ROOT / "schema_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                relevant = await session.call_tool("find_relevant_columns", {"db_id": db_id, "question": question})
                schema = await session.call_tool("get_schema", {"db_id": db_id})
                return {
                    "relevant_columns": [item.text for item in relevant.content],
                    "schema": schema.content[0].text
                }

writer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are the Writer Agent in VeriSQL.
Generate accurate SQLite SELECT queries using only the supplied database schema.
Use existing tables and columns, follow foreign keys for JOINs, never invent schema details,
never access sensitive columns, and generate only read-only SELECT queries.
State assumptions when the question is ambiguous.
{format_instructions}"""),
    ("human", """User question: {question}
Relevant columns: {relevant_columns}
Complete database schema: {schema}
Generate the SQL query.""")
])

writer_chain = writer_prompt | llm | sql_parser

async def generate_sql(question: str, db_id: str = "verisql_demo") -> SQLGeneration:
    context = await get_database_context(question, db_id)
    return writer_chain.invoke({
        "question": question,
        "relevant_columns": "\n".join(context["relevant_columns"]),
        "schema": context["schema"],
        "format_instructions": format_instructions
    })

async def repair_sql(question: str, previous_sql: str, evaluator_feedback: str, db_id: str = "verisql_demo") -> SQLGeneration:
    context = await get_database_context(question, db_id)
    repair_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are the VeriSQL Writer Agent.
Repair the previous SQL using the evaluator's feedback.
Use only the supplied SQLite schema. Generate one read-only SELECT query.
Do not repeat the identified mistake.
{format_instructions}"""),
        ("human", """Original question: {question}
Previous SQL: {previous_sql}
Evaluator feedback: {evaluator_feedback}
Relevant columns: {relevant_columns}
Database schema: {schema}
Generate corrected SQL.""")
    ])
    repair_chain = repair_prompt | llm | sql_parser
    return repair_chain.invoke({
        "question": question,
        "previous_sql": previous_sql,
        "evaluator_feedback": evaluator_feedback,
        "relevant_columns": "\n".join(context["relevant_columns"]),
        "schema": context["schema"],
        "format_instructions": format_instructions
    })

def validate_generated_sql(sql_query: str) -> dict:
    try:
        statements = parse(sql_query, read="sqlite")
        if len(statements) != 1 or statements[0] is None:
            return {"valid": False, "reason": "Only one SQL statement is allowed."}
        statement = statements[0]
        if not isinstance(statement, exp.Query):
            return {"valid": False, "reason": "Only read-only SELECT queries are allowed."}
        blocked = (exp.Insert, exp.Update, exp.Delete, exp.Create, exp.Drop, exp.Alter, exp.Command)
        if any(isinstance(node, blocked) for node in statement.walk()):
            return {"valid": False, "reason": "Write or schema-modification operations are not allowed."}
        return {"valid": True, "reason": "SQL passed structural validation."}
    except Exception as error:
        return {"valid": False, "reason": str(error)}

def log_repair(question: str, initial_sql: str, evaluator_feedback: str, repaired_sql: str, initial_validation: dict, final_validation: dict):
    log_entry = {
        "question": question,
        "initial_sql": initial_sql,
        "evaluator_feedback": evaluator_feedback,
        "repaired_sql": repaired_sql,
        "initial_validation": initial_validation,
        "final_validation": final_validation,
        "repair_success": final_validation["valid"]
    }
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with open(REPAIR_LOG_PATH, "a", encoding="utf-8") as file:
        file.write(json.dumps(log_entry) + "\n")
'''

WRITER_AGENT_PATH.write_text(writer_code, encoding="utf-8")

print("Writer Agent module created:", WRITER_AGENT_PATH.resolve())
print("File size:", WRITER_AGENT_PATH.stat().st_size, "bytes")

Writer Agent module created: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\agents\writer.py
File size: 5888 bytes


In [28]:
from agents.writer import generate_sql as standalone_generate_sql, validate_generated_sql as standalone_validate_sql

standalone_result = await standalone_generate_sql(
    "Find the names of employees working in the Engineering department."
)

print("GENERATED SQL:")
print(standalone_result.sql_query)

print("\nVALIDATION:")
print(standalone_validate_sql(standalone_result.sql_query))

GENERATED SQL:
SELECT e.employee_name FROM employees e JOIN departments d ON e.department_id = d.department_id WHERE d.department_name = 'Engineering';

VALIDATION:
{'valid': True, 'reason': 'SQL passed structural validation.'}


In [31]:
writer_server_code = r'''
import sys
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from mcp.server.fastmcp import FastMCP
from agents.writer import generate_sql as writer_generate_sql
from agents.writer import repair_sql as writer_repair_sql

mcp_server = FastMCP("VeriSQL Writer MCP")

@mcp_server.tool()
async def generate_sql(question: str, db_id: str = "verisql_demo") -> dict:
    """Generate SQLite SQL from a natural-language question."""
    result = await writer_generate_sql(question, db_id)
    return result.model_dump()

@mcp_server.tool()
async def repair_sql(question: str, previous_sql: str, evaluator_feedback: str, db_id: str = "verisql_demo") -> dict:
    """Repair SQL using Evaluator Agent feedback."""
    result = await writer_repair_sql(question, previous_sql, evaluator_feedback, db_id)
    return result.model_dump()

if __name__ == "__main__":
    mcp_server.run(transport="stdio")
'''

WRITER_SERVER_PATH.write_text(writer_server_code, encoding="utf-8")

print("Writer MCP server created:", WRITER_SERVER_PATH.resolve())
print("File size:", WRITER_SERVER_PATH.stat().st_size, "bytes")

Writer MCP server created: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\mcp_servers\writer\server.py
File size: 981 bytes


In [33]:
async def test_writer_mcp():
    server_params = StdioServerParameters(command=sys.executable, args=[str(WRITER_SERVER_PATH)])

    with open(PROJECT_ROOT / "writer_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                print("DISCOVERED WRITER MCP TOOLS:")
                for tool in tools.tools:
                    print(f"- {tool.name}: {tool.description}")

await test_writer_mcp()
async def test_writer_mcp():
    server_params = StdioServerParameters(command=sys.executable, args=[str(WRITER_SERVER_PATH)])

    with open(PROJECT_ROOT / "writer_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                print("DISCOVERED WRITER MCP TOOLS:")
                for tool in tools.tools:
                    print(f"- {tool.name}: {tool.description}")

await test_writer_mcp()

DISCOVERED WRITER MCP TOOLS:
- generate_sql: Generate SQLite SQL from a natural-language question.
- repair_sql: Repair SQL using Evaluator Agent feedback.
DISCOVERED WRITER MCP TOOLS:
- generate_sql: Generate SQLite SQL from a natural-language question.
- repair_sql: Repair SQL using Evaluator Agent feedback.


In [34]:
async def test_writer_mcp_tools():
    server_params = StdioServerParameters(command=sys.executable, args=[str(WRITER_SERVER_PATH)])

    with open(PROJECT_ROOT / "writer_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                print("TEST 1: GENERATE SQL")
                generation = await session.call_tool("generate_sql", {
                    "question": "Find employees working in the Engineering department."
                })
                print(generation.content[0].text)

                print("\nTEST 2: REPAIR SQL")
                repair = await session.call_tool("repair_sql", {
                    "question": "Find employees working in the Engineering department.",
                    "previous_sql": "SELECT e.name FROM employees e WHERE e.department_id = 1",
                    "evaluator_feedback": "Unknown column 'name'. Use 'employee_name' and join departments to filter Engineering."
                })
                print(repair.content[0].text)

await test_writer_mcp_tools()

TEST 1: GENERATE SQL
{
  "sql_query": "SELECT e.employee_id, e.employee_name\nFROM employees e\nJOIN departments d ON e.department_id = d.department_id\nWHERE d.department_name = 'Engineering';",
  "explanation": "This query retrieves the employee IDs and names of all employees whose department is named 'Engineering' by joining the employees and departments tables on the department_id foreign key and filtering on the department_name column.",
  "tables_used": [
    "employees",
    "departments"
  ],
  "assumptions": [
    "The department name to match is exactly 'Engineering' (case-sensitive).",
    "All employees have a department_id that correctly references the departments table."
  ]
}

TEST 2: REPAIR SQL
{
  "sql_query": "SELECT e.employee_name FROM employees e JOIN departments d ON e.department_id = d.department_id WHERE d.department_name = 'Engineering';",
  "explanation": "Joins employees with departments on department_id and filters for the Engineering department, returning e